# `c12_al` — Academic Libraries

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `AL2023`, `DRVAL2023` |
| Reference period | Fiscal year 2023 |
| Curated grain | `UNITID` |
| Output | `data/curated/c12_al.parquet` |

Library expenditure per student is a useful proxy for instructional support intensity, and it pairs naturally with the twelve-month enrollment denominator from c03 rather than a fall census count, because both are fiscal-year figures.

The expenditure hierarchy reconciles exactly, and the three `sums_to` rules below are therefore error-level rather than warnings: `LEXPTOT` is the sum of salaries and wages, **fringe benefits**, materials and services, and operations and maintenance. Omitting `LFRNGBN` is an easy mistake that breaks reconciliation for roughly two thirds of reporting institutions while leaving the other third apparently fine, which is exactly the kind of partial failure a hard rule catches and a spot check does not.

> **Pitfall.** The component is screener-gated, and the screener is visible in the data: LEXP100K records whether total library expenses reached $100,000, and institutions below that threshold skip the detailed expenditure items. A null in LEXMSTL therefore usually means not asked rather than zero. Imputing zeros would fabricate a large population of libraryless institutions that does not exist. Condition on LEXP100K before interpreting any detail item.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c12_al"
TABLES = ['AL2023', 'DRVAL2023']
GRAIN = ['UNITID']
REFERENCE_PERIOD = 'Fiscal year 2023'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,AL2023,266290,596f1330af705ade77363aa77b714a894470f1570550a7...,2026-09-24T17:19:39+00:00
1,DRVAL2023,74351,40f9f8cd05566885fee8d1988b6914dfdb1a4329a12b77...,2026-09-24T17:19:39+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'(2022-23|fiscal year 2023|FY2023)',
    table=TABLES[0],
)
print(intro[:600])

File documentation for Academic Library Data File: Fiscal year 2023
(Provisional release)
Filename AL2023
Overview The Academic Library survey became part of the Integrated Postsecondary Education Data system in collection year 2014-15.   Data include characteristics of the library, collections, expenditures and services. The number of full-time equivalent staff employed by the library were added this fiscal year.  Library data are only applicable for degree-granting institutions.  Degree-granting institutions with total expenditures over $100,000 dollars will have expenditure data. In 2016-17


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

38 variables documented, 11 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,LEXP100K,Were annual total library expenses greater tha...
2,LCOLELYN,Is the Library collection entirely electronic
3,LPBOOKS,Number of physical books
4,LEBOOKS,Number of digital/electronic books
5,LEDATAB,Number of digital/electronic databases
6,LPMEDIA,Number of physical media
7,LEMEDIA,Number of digital/electronic media
8,LPSERIA,Number of physical serials
9,LESERIA,Number of electronic serials


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'LEXP100K', 'LCOLELYN', 'LEXPTOT', 'LSALWAG', 'LFRNGBYN', 'LFRNGBN', 'LEXMSTL', 'LEXMSBB', 'LEXMSCS', 'LEXMSOT', 'LEXOMTL', 'LEXOMPS', 'LEXOMOT', 'LSTOTAL', 'LSLIBRN', 'LPBOOKS', 'LEBOOKS', 'LEDATAB', 'LTCLLCT', 'LTCRCLT']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (3695, 69)
schema: unchanged | added [] | removed []


,UNITID,LEXP100K,LCOLELYN,LEXPTOT,LSALWAG,LFRNGBYN,LFRNGBN,LEXMSTL,LEXMSBB,LEXMSCS,LEXMSOT,LEXOMTL,LEXOMPS,LEXOMOT,LSTOTAL,LSLIBRN,LPBOOKS,LEBOOKS,LEDATAB,LTCLLCT,LTCRCLT
0,100654,1,2,2706948.0,895925.0,1,287779.0,764780.0,29577.0,425257.0,309946.0,758464.0,17280.0,741184.0,26.00,9.0,367454.0,228749,107,795169,135329
1,100663,1,2,20005257.0,4541780.0,1,1803319.0,11029657.0,3488038.0,7309017.0,232602.0,2630501.0,10816.0,2619685.0,89.30,38.0,902731.0,836128,617,2004556,3170342
2,100690,1,2,145714.0,52655.0,2,0.0,87860.0,13048.0,67310.0,7502.0,5199.0,0.0,5199.0,1.00,1.0,50472.0,746,62,52500,29333
3,100706,1,2,3204968.0,1481428.0,2,0.0,1581745.0,47789.0,1506435.0,27521.0,141795.0,7126.0,134669.0,37.14,11.8,210388.0,859170,258,1454650,334140
4,100724,1,2,2763267.0,1667528.0,1,447847.0,373128.0,40359.0,332769.0,0.0,274764.0,5000.0,269764.0,49.00,13.0,224758.0,108091,189,375273,40321


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['LEXP100K', 'LCOLELYN']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 866 reserved-code cells across 20 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
37,XLEBOOKS,R,3600,0.9743
38,XLEBOOKS,Z,94,0.0254
39,XLEBOOKS,P,1,0.0003
40,XLEDATAB,R,3541,0.9583
41,XLEDATAB,Z,153,0.0414
42,XLEDATAB,P,1,0.0003
9,XLEXMSBB,R,2828,0.7654
10,XLEXMSBB,A,866,0.2344
11,XLEXMSBB,N,1,0.0003
12,XLEXMSCS,R,2828,0.7654


Columns under 90% reported — interpret with care:


column
XLEXMSBB    0.7654
XLEXMSCS    0.7654
XLEXMSOT    0.7654
XLEXMSTL    0.7654
XLEXOMOT    0.7654
XLEXOMPS    0.7654
XLEXOMTL    0.7654
XLEXPTOT    0.7654
XLFRNGBN    0.5283
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['LEXP100K', 'LCOLELYN']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,LEXP100K,LCOLELYN,LEXP100K_LABEL,LCOLELYN_LABEL
0,1,2,"Greater than or equal to $100,000",No
12,2,2,"Less than $100,000",No
20,2,1,"Less than $100,000",Yes
43,1,1,"Greater than or equal to $100,000",Yes


## 9. Reshape to the declared grain

Target grain: `UNITID`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID'] -> 3,695 rows, 0 duplicated


,UNITID,LEXP100K,LCOLELYN,LEXPTOT,LSALWAG,LFRNGBYN,LFRNGBN,LEXMSTL,LEXMSBB,LEXMSCS,LEXMSOT,LEXOMTL,LEXOMPS,LEXOMOT,LSTOTAL,LSLIBRN,LPBOOKS,LEBOOKS,LEDATAB,LTCLLCT,LTCRCLT,LEXP100K_LABEL,LCOLELYN_LABEL
0,100654,1,2,2706948.0,895925.0,1.0,287779.0,764780.0,29577.0,425257.0,309946.0,758464.0,17280.0,741184.0,26.00,9.0,367454.0,228749.0,107.0,795169.0,135329.0,"Greater than or equal to $100,000",No
1,100663,1,2,20005257.0,4541780.0,1.0,1803319.0,11029657.0,3488038.0,7309017.0,232602.0,2630501.0,10816.0,2619685.0,89.30,38.0,902731.0,836128.0,617.0,2004556.0,3170342.0,"Greater than or equal to $100,000",No
2,100690,1,2,145714.0,52655.0,2.0,0.0,87860.0,13048.0,67310.0,7502.0,5199.0,0.0,5199.0,1.00,1.0,50472.0,746.0,62.0,52500.0,29333.0,"Greater than or equal to $100,000",No
3,100706,1,2,3204968.0,1481428.0,2.0,0.0,1581745.0,47789.0,1506435.0,27521.0,141795.0,7126.0,134669.0,37.14,11.8,210388.0,859170.0,258.0,1454650.0,334140.0,"Greater than or equal to $100,000",No
4,100724,1,2,2763267.0,1667528.0,1.0,447847.0,373128.0,40359.0,332769.0,0.0,274764.0,5000.0,269764.0,49.00,13.0,224758.0,108091.0,189.0,375273.0,40321.0,"Greater than or equal to $100,000",No


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID'),
    iu.in_range('LEXPTOT', 0, None),
    iu.sums_to('LEXPTOT', ['LSALWAG', 'LFRNGBN', 'LEXMSTL', 'LEXOMTL'], tolerance=1),
    iu.sums_to('LEXMSTL', ['LEXMSBB', 'LEXMSCS', 'LEXMSOT'], tolerance=1),
    iu.sums_to('LEXOMTL', ['LEXOMPS', 'LEXOMOT'], tolerance=1),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,unique_key(UNITID),pass,0,0.0,Declared grain must be unique
1,"in_range(LEXPTOT,0,None)",pass,0,0.0,Value plausibility bound
2,sums_to(LEXPTOT),pass,0,0.0,Parts must reconcile within 1
3,sums_to(LEXMSTL),pass,0,0.0,Parts must reconcile within 1
4,sums_to(LEXOMTL),pass,0,0.0,Parts must reconcile within 1


PASSED


Report(table='c12_al', rows=3695, results=[{'name': 'unique_key(UNITID)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(LEXPTOT,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}, {'name': 'sums_to(LEXPTOT)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Parts must reconcile within 1', 'status': 'pass'}, {'name': 'sums_to(LEXMSTL)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Parts must reconcile within 1', 'status': 'pass'}, {'name': 'sums_to(LEXOMTL)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Parts must reconcile within 1', 'status': 'pass'}], generated_utc='2026-09-24T17:19:39+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='The component is screener-gated, and the screener is visible in the data: LEXP100K records whether total library expenses reached $100,000, and institutions below that threshold skip the detailed expenditure items. A null in LEXMSTL therefore usually means not asked rather than zero. Imputing zeros would fabricate a large population of libraryless institutions that does not exist. Condition on LEXP100K before interpreting any detail item.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c12_al.parquet (3,695 rows x 23 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. The component is screener-gated, and the screener is visible in the data: LEXP100K records whether total library expenses reached $100,000, and institutions below that threshold skip the detailed expenditure items. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.